## Creating ACS MIR occultation database

In this example, we show how to create a database including basic information about the ACS MIR solar occultation observations, so that it can be easily explored to find relevant occultations for our needs.

To create this database, we loop through each of the IKI geometry files and extract only the relevant information.

In [1]:
import acs
import pandas as pd
import glob,os
import numpy as np
from pathlib import Path
curr = os.getcwd()

### 1. Initialise the database

First, we just initialise an empty database object.

In [2]:
database = acs.mir.database.initialise_database()

print(database)

Empty DataFrame
Columns: [orbit, num, date, time, position, mtp, stp, IEflag, FPflag, Ls, LST, Latitude, Longitude, MY]
Index: []


### 2. Adding records

Now, we loop through each of the IKI geometry files and add the records to the database.

This code is adjustable, but we recommend having a folder structure similar to that stored in the IKI ftp site. This is:

- base_dir
    - MTP***
        - STP***
            - GEO_1_MIR_0B_ORB002257_N1_E_P1_05_F_SWP.txt
            - GEO_2_MIR_0B_ORB002257_N1_E_P1_05_F_SWP.txt
            - GEO_3_MIR_0B_ORB002257_N1_E_P1_05_F_SWP.txt
            - ...

In [3]:
iki_geometry_basedir = "/srv/workspace/data/tgo/acs/mir/iki/geometry/"

In [4]:
os.chdir(iki_geometry_basedir)
mtps = glob.glob('MTP*')
mtps = np.sort(mtps)

for imtp in range(len(mtps)):
    
    os.chdir(mtps[imtp])
    stps = glob.glob('STP*')
    stps = np.sort(stps)

    for istp in range(len(stps)):
        
        os.chdir(stps[istp])
        print(mtps[imtp],stps[istp])

        files = glob.glob('GEO_1*')
        files = np.sort(files)

        for ifile in range(len(files)):

            file = files[ifile]
            orb,nflag,date,time,IEobs,pos,FPobs,Ls,Loct,lat,lon = acs.mir.database.read_geometry_essentials(file)

            #Calculating the MY based on the date and time
            MY, Lsx, solx = acs.general.marstime.utc_to_marsdate(date+' '+time)
            if np.abs(Lsx-Ls)>0.1:
                print('error in orbit',orb,'num',nflag,' - Ls from marstime and ACS does not match',Ls,Lsx)
            else:
                database = acs.mir.database.add_record(database, orb, nflag, date, time, mtps[imtp], stps[istp], pos, IEobs, FPobs, Ls, Loct, lat, lon, MY[0])

        os.chdir('../')

    os.chdir('../')

MTP000 STP001
MTP000 STP002
MTP000 STP003
MTP000 STP004
MTP001 STP001
Added new record for orbit 1833, num 1.
Added new record for orbit 1839, num 1.
Added new record for orbit 1845, num 1.
Added new record for orbit 1849, num 1.
Added new record for orbit 1853, num 1.
Added new record for orbit 1857, num 1.
Added new record for orbit 1859, num 1.
Added new record for orbit 1863, num 1.
Added new record for orbit 1869, num 1.
Added new record for orbit 1871, num 1.
Added new record for orbit 1876, num 1.
Added new record for orbit 1879, num 1.
Added new record for orbit 1882, num 1.
Added new record for orbit 1888, num 1.
Added new record for orbit 1893, num 1.
Added new record for orbit 1896, num 1.
Added new record for orbit 1896, num 2.
Added new record for orbit 1900, num 1.
Added new record for orbit 1900, num 2.
Added new record for orbit 1903, num 1.
Added new record for orbit 1903, num 2.
MTP001 STP002
Added new record for orbit 1917, num 1.
Added new record for orbit 1917, num

/home/alday/Documents/Projects/acs-dist/acs/mir/database.py:175: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  database = pd.concat([database, pd.DataFrame([new_record])], ignore_index=True)


Added new record for orbit 2298, num 2.
Added new record for orbit 2302, num 1.
Added new record for orbit 2302, num 2.
Added new record for orbit 2304, num 1.
Added new record for orbit 2304, num 2.
Added new record for orbit 2307, num 1.
Added new record for orbit 2308, num 1.
Added new record for orbit 2309, num 1.
Added new record for orbit 2309, num 2.
Added new record for orbit 2311, num 1.
Added new record for orbit 2316, num 1.
Added new record for orbit 2319, num 1.
Added new record for orbit 2321, num 1.
Added new record for orbit 2326, num 1.
Added new record for orbit 2328, num 1.
Added new record for orbit 2333, num 1.
MTP002 STP007
Added new record for orbit 2344, num 1.
Added new record for orbit 2350, num 1.
Added new record for orbit 2356, num 1.
Added new record for orbit 2358, num 1.
Added new record for orbit 2361, num 1.
Added new record for orbit 2365, num 1.
Added new record for orbit 2369, num 1.
Added new record for orbit 2373, num 1.
Added new record for orbit

### 3. Writing output database

Now, we can easily write the database into a .csv file that easily readable with pandas.

Here, we are going to store it within the acs Python package data.

In [7]:
#Finding installation directory of ACS package
acs_dir = Path(acs.__file__).resolve().parent
database_file = (acs_dir.parent / "data" / "mir_database.csv").resolve()

In [9]:
acs.mir.database.write_database(database,database_file)

### 4. Searching records within database

We can use this database object to explore the database. For instance, we can search particular orbits to see their basic characteristics.

In [12]:
orbits = [2005,2005,22270,22270]
nums = [1,2,1,2]

In [14]:
records = acs.mir.database.search_records(database, orbits, nums)

print(records)

      orbit num        date      time position     mtp     stp IEflag FPflag  \
59     2005   1  2018-05-06  05:49:16       05  MTP001  STP003      I      F   
60     2005   2  2018-05-06  06:10:32       12  MTP001  STP003      E      F   
7645  22270   1  2022-11-20  07:53:41       07  MTP060  STP240      I      F   
7646  22270   2  2022-11-20  08:27:38       05  MTP060  STP240      E      F   

          Ls        LST  Latitude  Longitude  MY  
59    170.90  17.932222  -12.4385  -138.8853  34  
60    170.91  18.432778   60.4382  -136.5523  34  
7645  341.42  18.310556  -31.7205    -1.9971  36  
7646  341.44  12.553333   81.9675   -96.6206  36  
